In [1]:
from sleap_nn.predict import run_inference
import glob
import tqdm
import re
import os
import sleap_io as sio
import pandas as pd

In [2]:
# Functions to sort file names correctly
def atoi(text):
    return int(text) if text.isdigit() else text

def natural_keys(text):
    return [atoi(c) for c in re.split(r'(\d+)', text)]

In [3]:
# Multi head sleap model
centered_instance_model_path = r"D:\sleap-3D\talmo\models\headmic_model\centered_instance_unet_body_1-15"
centroid_model_path = r"D:\sleap-3D\talmo\models\headmic_model\centroid_unet_body_1_anchor-15"

In [ ]:
for exp in [438]:
    
    experiment_no = exp
    folder = f"D:/big_setup/experiment_{experiment_no}/concatenated_data_cam_mic_sync/"
    save_path = folder+"sleap_predictions/"
    os.makedirs(save_path,exist_ok=True)

    vid_files = glob.glob(folder + "*center*.mp4")
    vid_files.sort(key=natural_keys)



    print("Tracking")
    for file in tqdm.tqdm(vid_files):
        
        # Initial
        if any(i in file for i in []): #"000","001","002","003","004","005","006","007","008","009","010","011","012","013","014","015","016","017"

            print(f"Not running:{file}")
            continue

        else:
            print(f"Running:{file}")

            preds = run_inference(
            data_path=file,
            model_paths=[centroid_model_path, centered_instance_model_path],
            device='cuda',
            make_labels=True,
            output_path=save_path+file.split("\\")[-1]+".pred_body_1_anchor.slp",
            tracking=True,
            candidates_method="local_queues",
            max_tracks=2, #optional
            )
            


        # Second run 
        # if any(i in file for i in ["004","018","020","028","029"]):
            

        #     print(f"Running:{file}")

        #     preds = run_inference(
        #     data_path=file,
        #     model_paths=[centroid_model_path, centered_instance_model_path],
        #     device='cuda',
        #     make_labels=True,
        #     output_path=save_path+file.split("\\")[-1]+".pred_body_1_anchor.slp",
        #     tracking=True,
        #     candidates_method="local_queues",
        #     max_tracks=2, #optional
        #     )
        # else:
        #     print(f"Not running:{file}")
        #     continue
    

    print(f"Done with experiment: {exp}")


Tracking


  0%|          | 0/35 [00:00<?, ?it/s]

Running:D:/big_setup/experiment_438/concatenated_data_cam_mic_sync\video_gily_center_000.mp4
2026-03-18 15:42:57 | INFO | sleap_nn.predict:run_inference:319 | Started inference at: 2026-03-18 15:42:57.193886
2026-03-18 15:42:57 | INFO | sleap_nn.predict:run_inference:335 | Using device: cuda


Output()

### Making csv files from outputs

In [17]:
for exp in [388]:
    folder = f"D:/big_setup/experiment_{exp}/concatenated_data_cam_mic_sync/"
    labels_path = folder+"sleap_predictions/"
    tracks_path = folder+"tracks/"
    os.makedirs(tracks_path,exist_ok=True)

    labels = glob.glob(labels_path+"*.slp")
    for label in tqdm.tqdm(labels):
        preds = sio.load_file(label)
        tmp_str = label.split("\\")[-1].split(".")[0]
        sio.save_csv(preds,tracks_path+"labels.body_1_anchor."+tmp_str+".analysis.csv",save_metadata= True)
        # data = sio.load_csv(tracks_path+"labels.body_1_anchor."+tmp_str+".analysis.csv")
        # data.save(tracks_path+"labels.body_1_anchor."+tmp_str+".analysis.csv.slp")
        
    print(f"Experiment {exp} done")


100%|██████████| 26/26 [00:08<00:00,  2.94it/s]

Experiment 388 done


In [ ]:
# for exp in [433]:
#     folder = f"D:/big_setup/experiment_{exp}/concatenated_data_cam_mic_sync/"
#     labels_path = folder+"sleap_predictions/"
#     tracks_path = folder+"tracks/"
#     os.makedirs(tracks_path,exist_ok=True)

#     labels = glob.glob(labels_path+"*.slp")
#     for label in tqdm.tqdm(labels):
#         preds = sio.load_file(label)
#         data = {}
#         for frame in preds.labeled_frames:
#             for instance in frame.instances:
#                 if len(data) == 0:
#                     data["track"] = [instance.track.name]
#                     data["frame_idx"] = [frame.frame_idx]
#                     data["instance.score"] = [instance.score]
#                     for entry in instance.points:
#                         data[f"{entry[-1]}.x"] = [entry[0][0]]
#                         data[f"{entry[-1]}.y"] = [entry[0][1]]
#                         data[f"{entry[-1]}.score"] = [entry[1]]
#                 else:
#                     data["track"].append(instance.track.name)
#                     data["frame_idx"].append(frame.frame_idx)
#                     try:
#                         data["instance.score"].append(instance.score)
#                     except:
#                         data["instance.score"].append() 
                        
                        
#                     for entry in instance.points:
#                         data[f"{entry[-1]}.x"].append(entry[0][0])
#                         data[f"{entry[-1]}.y"].append(entry[0][1])
#                         data[f"{entry[-1]}.score"].append(entry[1])
    
#         df = pd.DataFrame.from_dict(data)

#         tmp_str = label.split("\\")[-1].split(".")[0]
#         idx = tmp_str.split('_')[-1]
#         df['track'] = f"exp_{str(exp)}_idx_{idx}_" + df['track']
#         df.to_csv(tracks_path+"labels.body_1_anchor."+tmp_str+".analysis.csv",index=False)
        
#     print(f"Experiment {exp} done")
